# Logistic Reduced-Rank Regression (MM) — Reproducible Notebook

Contains MM algorithm implementation, toy example, UCI preprocessing, triplots, evaluation and benchmarks.

In [1]:

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, time
from sklearn.linear_model import LogisticRegression
sns.set_style('whitegrid')
print('Imports ready.')


Imports ready.


In [2]:

def logistic_rrr_MM(X, Y, S, maxiter=200, tol=1e-6, verbose=False):
    N, P = X.shape; R = Y.shape[1]
    B = np.random.randn(P, S) * 0.01
    V = np.random.randn(R, S) * 0.01
    p = np.clip(Y.mean(axis=0), 1e-6, 1-1e-6)
    m = np.log(p/(1-p))
    XtX = X.T @ X
    eigvals, eigvecs = np.linalg.eigh(XtX)
    eigvals = np.clip(eigvals, 1e-12, None)
    inv_sqrt = eigvecs @ np.diag(1.0/np.sqrt(eigvals)) @ eigvecs.T
    sqrt_XtX = eigvecs @ np.diag(np.sqrt(eigvals)) @ eigvecs.T
    prev_ll = -np.inf
    for it in range(maxiter):
        lin = X @ (B @ V.T) + np.ones((N,1)) @ m.reshape(1,R)
        pi = 1.0/(1.0+np.exp(-lin))
        Z = lin + 4.0*(Y - pi)
        D = Z - X @ (B @ V.T)
        m = D.mean(axis=0)
        U = Z - np.ones((N,1)) @ m.reshape(1,R)
        M = inv_sqrt @ (X.T @ U)
        Pmat, svals, Qt = np.linalg.svd(M, full_matrices=False)
        P_S = Pmat[:, :S]
        Q_S = Qt[:S, :].T
        B = np.sqrt(N) * (inv_sqrt @ P_S)
        V = (1.0/np.sqrt(N)) * (sqrt_XtX @ Q_S)
        lin = X @ (B @ V.T) + np.ones((N,1)) @ m.reshape(1,R)
        pi = np.clip(1.0/(1.0+np.exp(-lin)), 1e-12, 1-1e-12)
        ll = np.sum(Y * np.log(pi) + (1-Y) * np.log(1-pi))
        if np.abs(ll - prev_ll) < tol:
            break
        prev_ll = ll
    return B, V, m
print('MM function defined.')


MM function defined.
